# bias over time across languages
is the bias trend English-specific vs robust?
- are the bias time-trends we see in English stable when we move to other languages?


### bias over time across languages with CrowS-Pairs (English/French)
- compute the same causal-LM adaptions as in the other notebooks, but separately per language
- outputs bias_crows_pairs_multilingual.csv

In [1]:
# imports
import os, gc, json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from difflib import SequenceMatcher
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import GPTNeoForCausalLM 

In [ ]:
# config
OUT_DIR = "results_fairness_over_time"
os.makedirs(OUT_DIR, exist_ok=True)

SUMMARY_CSV = os.path.join(OUT_DIR, "summary_all_models.csv")  # existing models list
BIAS_ML_CSV = os.path.join(OUT_DIR, "bias_crows_pairs_multilingual.csv")

LANGS = ["english", "french"]  # HF dataset configs
BATCH_SIZE = 8
MAX_LEN = 128

In [3]:
# load multilingual CrowS-Pairs
from datasets import load_dataset

def load_crows_pairs_hf(subset: str):  # subset in {"english", "french"}
    ds = load_dataset("jannalu/crows_pairs_multilingual", subset, split="test")
    df = ds.to_pandas()
    # columns (from HF): sent_more, sent_less, stereo_antistereo, bias_type
    return df, "sent_more", "sent_less", "bias_type"

df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR = load_crows_pairs_hf("french")
print(df_fr.shape, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR)

df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN = load_crows_pairs_hf("english")
print(df_en.shape, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN)

(1677, 4) sent_more sent_less bias_type
(1677, 4) sent_more sent_less bias_type


In [4]:
# word-level overlap masks (unmodified tokens)
def unmodified_word_masks(words_a, words_b):
    sm = SequenceMatcher(a=words_a, b=words_b)
    mask_a = [False] * len(words_a)
    mask_b = [False] * len(words_b)
    for i, j, n in sm.get_matching_blocks():
        if n == 0:
            continue
        for k in range(n):
            mask_a[i + k] = True
            mask_b[j + k] = True
    return mask_a, mask_b


In [5]:
# batched causal-LM scoring on umodified tokens
@torch.inference_mode()
def score_sentences_unmodified_causal(tokenizer, model, batch_word_lists, batch_unmod_word_masks, max_len=128):
    if not getattr(tokenizer, "is_fast", False):
        raise ValueError("Need a fast tokenizer for word_ids mapping (use_fast=True).")

    enc = tokenizer(
        batch_word_lists,
        is_split_into_words=True,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_len,
        add_special_tokens=True,
    )
    dev = model.get_input_embeddings().weight.device
    input_ids = enc["input_ids"].to(dev)
    attn = enc["attention_mask"].to(dev)

    out = model(input_ids=input_ids, attention_mask=attn, use_cache=False)
    logits = out.logits  # [B,T,V]

    # teacher-forced log-likelihood
    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]
    logprobs = F.log_softmax(shift_logits, dim=-1)
    token_logp = logprobs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)  # [B,T-1]

    B, Tm1 = token_logp.shape
    keep = torch.zeros((B, Tm1), dtype=torch.bool, device=dev)

    for b in range(B):
        word_ids = enc.word_ids(batch_index=b)  # length T
        word_ids = word_ids[1:]                 # align to T-1
        unmod_mask_words = batch_unmod_word_masks[b]

        flags = []
        for wid in word_ids:
            if wid is None:
                flags.append(False)
            else:
                flags.append(bool(unmod_mask_words[wid]))
        keep[b, :len(flags)] = torch.tensor(flags, device=dev, dtype=torch.bool)

    scores = (token_logp * keep).sum(dim=1)  # [B]
    return scores.detach().cpu().numpy()


def compute_crows_bias_causal(tokenizer, model, df_pairs, more_col, less_col, cat_col=None,
                             batch_size=8, max_len=128):
    prefer_more = []
    cats = []

    sent_more_words, sent_less_words, mask_more, mask_less = [], [], [], []
    for _, row in df_pairs.iterrows():
        s_more = str(row[more_col])
        s_less = str(row[less_col])
        w_more = s_more.split()
        w_less = s_less.split()
        m_more, m_less = unmodified_word_masks(w_more, w_less)

        sent_more_words.append(w_more)
        sent_less_words.append(w_less)
        mask_more.append(m_more)
        mask_less.append(m_less)

        cats.append(str(row[cat_col]) if cat_col is not None else "all")

    N = len(sent_more_words)
    for s in range(0, N, batch_size):
        bm_words = sent_more_words[s:s+batch_size]
        bl_words = sent_less_words[s:s+batch_size]
        bm_mask  = mask_more[s:s+batch_size]
        bl_mask  = mask_less[s:s+batch_size]

        sc_more = score_sentences_unmodified_causal(tokenizer, model, bm_words, bm_mask, max_len=max_len)
        sc_less = score_sentences_unmodified_causal(tokenizer, model, bl_words, bl_mask, max_len=max_len)

        prefer_more.extend((sc_more > sc_less).tolist())

    prefer_more = np.array(prefer_more, dtype=int)
    bias_overall = 100.0 * prefer_more.mean()

    per_cat = None
    if cat_col is not None:
        per_cat = {}
        cats_arr = np.array(cats)
        for c in sorted(set(cats_arr)):
            idx = (cats_arr == c)
            if idx.sum() == 0:
                per_cat[c] = np.nan
            else:
                per_cat[c] = 100.0 * prefer_more[idx].mean()

    return bias_overall, per_cat

In [6]:
# model loading, same 
def load_model_and_tokenizer_causal(model_id, trust_remote_code=False):
    tok = AutoTokenizer.from_pretrained(model_id, use_fast=True, trust_remote_code=trust_remote_code)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb,
        device_map={"": 0} if torch.cuda.is_available() else "cpu",
        torch_dtype=torch.float16,
        trust_remote_code=trust_remote_code,
    )
    model.eval()
    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tok.pad_token_id
    return tok, model


def cleanup(model, tok):
    del model
    del tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [7]:
# CSV upsert utils (year, model_id, lang)
def read_csv_if_exists(path):
    return pd.read_csv(path) if os.path.exists(path) else None

def already_ran_bias_lang(year: int, model_id: str, lang: str) -> bool:
    if not os.path.exists(BIAS_ML_CSV):
        return False
    df = pd.read_csv(BIAS_ML_CSV)
    if df.empty:
        return False
    mask = (
        (df["year"].astype(int) == int(year)) &
        (df["model_id"].astype(str) == str(model_id)) &
        (df["lang"].astype(str) == str(lang))
    )
    return bool(mask.any())

def upsert_bias_lang_row(row: dict):
    df_old = read_csv_if_exists(BIAS_ML_CSV)
    df_new = pd.DataFrame([row])

    if df_old is None:
        df_out = df_new
    else:
        mask_keep = ~(
            (df_old["year"].astype(int) == int(row["year"])) &
            (df_old["model_id"].astype(str) == str(row["model_id"])) &
            (df_old["lang"].astype(str) == str(row["lang"]))
        )
        df_out = pd.concat([df_old.loc[mask_keep], df_new], ignore_index=True)

    df_out = df_out.sort_values(["year", "model_id", "lang"]).reset_index(drop=True)
    df_out.to_csv(BIAS_ML_CSV, index=False)
    return df_out

In [8]:
def run_one_model_bias_lang(model_spec: dict, lang: str, df_pairs: pd.DataFrame,
                            more_col: str, less_col: str, cat_col: str | None,
                            force: bool = False, batch_size: int = 8, max_len: int = 128):

    year = int(model_spec["year"])
    model_id = str(model_spec["id"])
    trust_remote_code = bool(model_spec.get("trust_remote_code", False))

    if (not force) and already_ran_bias_lang(year, model_id, lang):
        print(f"SKIP (already in {BIAS_ML_CSV}): {year}  {model_id}  {lang} | set force=True to rerun")
        return None

    print("\n" + "=" * 80)
    print(f"RUN BIAS {year}  {model_id}  lang={lang}")
    print("=" * 80)

    tok, model = load_model_and_tokenizer_causal(model_id, trust_remote_code=trust_remote_code)

    bias, bias_by_cat = compute_crows_bias_causal(
        tok, model, df_pairs, more_col, less_col, cat_col=cat_col,
        batch_size=batch_size, max_len=max_len
    )

    row = {
        "year": year,
        "model_id": model_id,
        "lang": lang,
        "crows_bias": float(bias),
    }
    if bias_by_cat is not None:
        row["crows_bias_by_cat_json"] = json.dumps(bias_by_cat)

    upsert_bias_lang_row(row)
    print("Saved/updated:", BIAS_ML_CSV)

    cleanup(model, tok)
    return row

In [9]:
# french
spec = {"year": 2021, "id": "EleutherAI/gpt-neo-2.7B", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2021  EleutherAI/gpt-neo-2.7B  french | set force=True to rerun


In [10]:
# english
spec = {"year": 2021, "id": "EleutherAI/gpt-neo-2.7B", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2021  EleutherAI/gpt-neo-2.7B  english | set force=True to rerun


In [11]:
# french
spec = {"year": 2022, "id": "bigscience/bloom-3b", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2022  bigscience/bloom-3b  french | set force=True to rerun


In [12]:
# english
spec = {"year": 2022, "id": "bigscience/bloom-3b", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2022  bigscience/bloom-3b  english | set force=True to rerun


In [13]:
# french
spec = {"year": 2023, "id": "stabilityai/stablelm-3b-4e1t", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2023  stabilityai/stablelm-3b-4e1t  french | set force=True to rerun


In [14]:
# english
spec = {"year": 2023, "id": "stabilityai/stablelm-3b-4e1t", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2023  stabilityai/stablelm-3b-4e1t  english | set force=True to rerun


In [15]:
# french
spec = {"year": 2023, "id": "EleutherAI/pythia-2.8b", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2023  EleutherAI/pythia-2.8b  french | set force=True to rerun


In [16]:
# english
spec = {"year": 2023, "id": "EleutherAI/pythia-2.8b", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2023  EleutherAI/pythia-2.8b  english | set force=True to rerun


In [17]:
# french
spec = {"year": 2023, "id": "stabilityai/stablelm-3b-4e1t", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2023  stabilityai/stablelm-3b-4e1t  french | set force=True to rerun


In [18]:
# english
spec = {"year": 2023, "id": "stabilityai/stablelm-3b-4e1t", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2023  stabilityai/stablelm-3b-4e1t  english | set force=True to rerun


In [19]:
# french
spec = {"year": 2024, "id": "Qwen/Qwen2.5-3B", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2024  Qwen/Qwen2.5-3B  french | set force=True to rerun


In [20]:
# english
spec = {"year": 2024, "id": "Qwen/Qwen2.5-3B", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2024  Qwen/Qwen2.5-3B  english | set force=True to rerun


In [21]:
# french
spec = {"year": 2024, "id": "tiiuae/Falcon3-3B-Base", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2024  tiiuae/Falcon3-3B-Base  french | set force=True to rerun


In [22]:
# english
spec = {"year": 2024, "id": "tiiuae/Falcon3-3B-Base", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2024  tiiuae/Falcon3-3B-Base  english | set force=True to rerun


In [23]:
# french
spec = {"year": 2025, "id": "HuggingFaceTB/SmolLM3-3B-Base", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2025  HuggingFaceTB/SmolLM3-3B-Base  french | set force=True to rerun


In [24]:
# english
spec = {"year": 2025, "id": "HuggingFaceTB/SmolLM3-3B-Base", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2025  HuggingFaceTB/SmolLM3-3B-Base  english | set force=True to rerun


In [25]:
# french
spec = {"year": 2022, "id": "facebook/opt-2.7b", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2022  facebook/opt-2.7b  french | set force=True to rerun


In [26]:
# english
spec = {"year": 2022, "id": "facebook/opt-2.7b", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2022  facebook/opt-2.7b  english | set force=True to rerun


In [27]:
# french
spec = {"year": 2023, "id": "togethercomputer/RedPajama-INCITE-Base-3B-v1", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2023  togethercomputer/RedPajama-INCITE-Base-3B-v1  french | set force=True to rerun


In [28]:
# english
spec = {"year": 2023, "id": "togethercomputer/RedPajama-INCITE-Base-3B-v1", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2023  togethercomputer/RedPajama-INCITE-Base-3B-v1  english | set force=True to rerun


In [29]:
# french
spec = {"year": 2023, "id": "openlm-research/open_llama_3b", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2023  openlm-research/open_llama_3b  french | set force=True to rerun


In [30]:
# english
spec = {"year": 2023, "id": "openlm-research/open_llama_3b", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2023  openlm-research/open_llama_3b  english | set force=True to rerun


In [31]:
# french
spec = {"year": 2023, "id": "microsoft/phi-2", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2023  microsoft/phi-2  french | set force=True to rerun


In [32]:
# english
spec = {"year": 2023, "id": "microsoft/phi-2", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2023  microsoft/phi-2  english | set force=True to rerun


In [33]:
# french
spec = {"year": 2023, "id": "cerebras/Cerebras-GPT-2.7B", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2023  cerebras/Cerebras-GPT-2.7B  french | set force=True to rerun


In [34]:
# english
spec = {"year": 2023, "id": "cerebras/Cerebras-GPT-2.7B", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2023  cerebras/Cerebras-GPT-2.7B  english | set force=True to rerun


In [35]:
# french
spec = {"year": 2025, "id": "LiquidAI/LFM2-2.6B", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2025  LiquidAI/LFM2-2.6B  french | set force=True to rerun


In [36]:
# english
spec = {"year": 2025, "id": "LiquidAI/LFM2-2.6B", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2025  LiquidAI/LFM2-2.6B  english | set force=True to rerun


In [37]:
# french
spec = {"year": 2025, "id": "Salesforce/xLAM-2-3b-fc-r", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2025  Salesforce/xLAM-2-3b-fc-r  french | set force=True to rerun


In [38]:
# english
spec = {"year": 2025, "id": "Salesforce/xLAM-2-3b-fc-r", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2025  Salesforce/xLAM-2-3b-fc-r  english | set force=True to rerun


In [39]:
# french
spec = {"year": 2022, "id": "bigscience/bloomz-3b", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2022  bigscience/bloomz-3b  french | set force=True to rerun


In [40]:
# english
spec = {"year": 2022, "id": "bigscience/bloomz-3b", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2022  bigscience/bloomz-3b  english | set force=True to rerun


In [41]:
# french
spec = {"year": 2023, "id": "openlm-research/open_llama_3b_v2", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2023  openlm-research/open_llama_3b_v2  french | set force=True to rerun


In [42]:
# english
spec = {"year": 2023, "id": "openlm-research/open_llama_3b_v2", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2023  openlm-research/open_llama_3b_v2  english | set force=True to rerun


In [43]:
# french
spec = {"year": 2024, "id": "state-spaces/mamba-2.8b-hf", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2024  state-spaces/mamba-2.8b-hf  french | set force=True to rerun


In [44]:
# english
spec = {"year": 2024, "id": "state-spaces/mamba-2.8b-hf", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2024  state-spaces/mamba-2.8b-hf  english | set force=True to rerun


In [45]:
# french
spec = {"year": 2025, "id": "LiquidAI/LFM2-2.6B", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2025  LiquidAI/LFM2-2.6B  french | set force=True to rerun


In [46]:
# english
spec = {"year": 2025, "id": "LiquidAI/LFM2-2.6B", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2025  LiquidAI/LFM2-2.6B  english | set force=True to rerun


In [47]:
# french
spec = {"year": 2021, "id": "facebook/xglm-2.9B", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2021  facebook/xglm-2.9B  french | set force=True to rerun


In [48]:
# english
spec = {"year": 2021, "id": "facebook/xglm-2.9B", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2021  facebook/xglm-2.9B  english | set force=True to rerun


In [49]:
# french
spec = {"year": 2022, "id": "EleutherAI/pythia-2.8b-v0",  "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2022  EleutherAI/pythia-2.8b-v0  french | set force=True to rerun


In [50]:
# english
spec = {"year": 2022, "id": "EleutherAI/pythia-2.8b-v0",  "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2022  EleutherAI/pythia-2.8b-v0  english | set force=True to rerun


In [51]:
# french
spec = {"year": 2022, "id": "EleutherAI/pythia-2.8b-deduped-v0", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2022  EleutherAI/pythia-2.8b-deduped-v0  french | set force=True to rerun


In [52]:
# english
spec = {"year": 2022, "id": "EleutherAI/pythia-2.8b-deduped-v0", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2022  EleutherAI/pythia-2.8b-deduped-v0  english | set force=True to rerun


In [53]:
# french
spec = {"year": 2024, "id": "state-spaces/mamba-2.8b-hf", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2024  state-spaces/mamba-2.8b-hf  french | set force=True to rerun


In [54]:
# english
spec = {"year": 2024, "id": "state-spaces/mamba-2.8b-hf", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2024  state-spaces/mamba-2.8b-hf  english | set force=True to rerun


In [55]:
# french
spec = {"year": 2022, "id": "Salesforce/codegen-2B-multi", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2022  Salesforce/codegen-2B-multi  french | set force=True to rerun


In [56]:
# english
spec = {"year": 2022, "id": "Salesforce/codegen-2B-multi", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2022  Salesforce/codegen-2B-multi  english | set force=True to rerun


In [57]:
# french
spec = {"year": 2024, "id": "ibm-granite/granite-3.1-3B-A800M-Base", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2024  ibm-granite/granite-3.1-3B-A800M-Base  french | set force=True to rerun


In [58]:
# english
spec = {"year": 2024, "id": "ibm-granite/granite-3.1-3B-A800M-Base", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2024  ibm-granite/granite-3.1-3B-A800M-Base  english | set force=True to rerun


In [59]:
# french
spec = {"year": 2024, "id": "stanford-crfm/BioMedLM", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2024  stanford-crfm/BioMedLM  french | set force=True to rerun


In [60]:
# english
spec = {"year": 2024, "id": "stanford-crfm/BioMedLM", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2024  stanford-crfm/BioMedLM  english | set force=True to rerun


In [61]:
# french
spec = {"year": 2024, "id": "stabilityai/stable-code-3b", "trust_remote_code": False}
run_one_model_bias_lang(spec, "french", df_fr, MORE_COL_FR, LESS_COL_FR, CAT_COL_FR,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2024  stabilityai/stable-code-3b  french | set force=True to rerun


In [62]:
# english
spec = {"year": 2024, "id": "stabilityai/stable-code-3b", "trust_remote_code": False}
run_one_model_bias_lang(spec, "english", df_en, MORE_COL_EN, LESS_COL_EN, CAT_COL_EN,
                        force=False, batch_size=2, max_len=128)

SKIP (already in results_fairness_over_time\bias_crows_pairs_multilingual.csv): 2024  stabilityai/stable-code-3b  english | set force=True to rerun
